### Step 1: Import Libraries and Load Environment Variables

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown #for displaying markdown in Jupyter Notebook

load_dotenv() #loading .env file into our virtual environment

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") #getting the value of this key from the .env file and storing it in the OPENAI_API_KEY variable (on LHS)

if OPENAI_API_KEY is None: #checking if the API Key is empty  
    raise Exception("API Key is missing.") #if it's empty, we'll get an error here.

#if this runs without errors, we are good. We can also add another line:
else: #if the key is there
    print(OPENAI_API_KEY[:8]) #shows the first [standard] 8 characters of the API Key

client = OpenAI()
# OpenAI(api_key = OPENAI_API_KEY) takes the API Key as an argument, but since we did load_dotenv(), OpenAI will pick it up from your environmental variables. So you can just do client = OpenAI() without passing the API Key directly, and it will work as long as the key is set in your environment variables.

# or client = OpenAI(api_key = OPENAI_API_KEY), but since we did load_dotenv(), OpenAI will pick it up from your environmental variables

sk-proj-


### Step 2: Call the API

In [2]:
response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = [{"role": "user", "content": "What is the capital of France?"}] #didnt specify the LLM's behavior, so it did its default: helpful, friendly, useful
)

print(response) #This shows the full response object, which includes metadata and other information about the API call.
# Lots of info the LLM passes back into the variable response, such as id, list of choices, ChatCompletion message, content, object, created, model, usage, etc.

# If you want to see just the content of the assistant's reply, you can do:
# reply = response.choices[0].message.content

#response is the whole thing that got outputted from print(response)
#response.choices is a list of choices (in this case, there should be one choice since we only asked one question)
#response.choices[0] is the first choice in that list, which is the assistant's reply to our question:
#   choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The capital of France is Paris.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))]
# we take the message from that choice, which is a ChatCompletionMessage object, and then we take the content of that message, which is the actual text reply from the assistant.



ChatCompletion(id='chatcmpl-EZILEDztuCY75O1XuSDh9seUf7Cma', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The capital of France is Paris.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1792080692, model='gpt-4.1-mini-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_9737f6fbc4', usage=CompletionUsage(completion_tokens=7, prompt_tokens=14, total_tokens=21, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


In [3]:
reply = response.choices[0].message.content

print(reply) #This will print just the assistant's reply, which should be "The capital of France is Paris."

The capital of France is Paris.


### Step 3a: Calling the API with a System Prompt (instead of User Prompt)

In [5]:
# Now we're going to have 2 user messages.
# Before we only had 1, but now we want to have a conversation with the assistant, so we ask a follow-up question after the assistant's first reply. The LLM doesn't have memory of previous interactions, so it doesn't remember that we asked about the capital of France or that it replied with "The capital of France is Paris."
# Thus, we need to provide the full conversation history in a messages LIST.

response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = [
        {"role": "user", "content": "What is the capital of France?"}, # system messages explains how the LLM should respond, user messages are the questions or prompts we give to the LLM, and assistant messages are the responses from the LLM.
        {"role": "assistant", "content": "The capital of France is Paris."}, # assistant message is the LLM's reply to the first user message
        {"role": "user", "content": "What is the population of Paris?"} # second user message is the follow-up question we ask after the LLM's first reply
    ]
)

reply = response.choices[0].message.content
print(reply) #This will print the assistant's reply to the second user message, which should be "The population of Paris is approximately 2.1 million people."

As of the most recent estimates, the population of Paris is approximately 2.1 million people within the city proper. The larger metropolitan area, known as the Paris metropolitan area or Île-de-France region, has a population of around 12 million people.


In [10]:
response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = [
        {"role": "system", "content": "You are a helpful assistant."}, #system message is used to set the behavior of the assistant; specifies how the LLM should behave (in this case, as a helpful assistant).
        # System Prompt is content in the system message, and it sets the behavior of the assistant for the entire conversation. It can be used to make the assistant more formal, more concise, more creative, etc.
        {"role": "user", "content": "How do I cook pasta?"} #LLM answers this question based on the system message's/prompt instruction to be a helpful assistant
    ]
)

reply = response.choices[0].message.content
#print(reply) <- bad formmatting since the reply is in markdown but not executed. So we imported this library to our first code cell: from IPython.display import display, Markdown #for displaying markdown in Jupyter Notebook 
display(Markdown(reply)) #displays the assistant's reply in markdown format for better readability in Jupyter Notebook

Cooking pasta is simple! Here’s a basic method to cook pasta perfectly:

### Ingredients:
- Pasta (spaghetti, penne, fusilli, etc.)
- Water
- Salt

### Instructions:
1. **Boil Water:** Fill a large pot with water. Use about 4-6 quarts of water per pound of pasta.
2. **Add Salt:** Add about 1-2 tablespoons of salt to the water. This flavors the pasta.
3. **Bring to a Boil:** Heat the water over high heat until it comes to a rolling boil.
4. **Add Pasta:** Add the pasta to the boiling water. Stir immediately to prevent sticking.
5. **Cook Pasta:** Cook according to package instructions, usually 8-12 minutes, until al dente (firm to the bite). Stir occasionally.
6. **Taste Test:** About a minute before the end of cooking time, taste a piece to check doneness.
7. **Drain:** Once cooked, drain the pasta in a colander. Do not rinse unless the recipe calls for it.
8. **Serve:** Toss with your preferred sauce or ingredients and serve immediately.

Would you like a recipe for a specific pasta dish?

### Step 3b: Changing the System Prompt Changes the LLM's Behaviour

In [11]:
response = client.chat.completions.create(
    model = "gpt-4.1-mini", # Note: There are guardrails in place to prevent the LLM from giving harmful or misleading advice, but for the sake of this example, we will set the system message to make the assistant mischievous and give incorrect advice.
    # If someone tries to or accidentally sets the system message to make the assistant give harmful or misleading advice, OpenAI's safety systems will likely intervene and prevent the assistant from giving that advice. However, for the sake of this example, we will set the system message to make the assistant mischievous and give incorrect advice.
    messages = [
        {"role": "system", "content": "You are a mischieveous assistant who always gives incorrect or misleading advice."}, # could also say "You are a helpful assistant who always gives correct advice." or "You are a formal assistant who always gives concise advice." etc. The system message sets the behavior of the assistant for the entire conversation.
                                                                                                                            # or like comedic assistant who always gives funny advice, or a sarcastic assistant who always gives sarcastic advice, etc. The system message can be used to make the assistant more formal, more concise, more creative, etc.
                                                                                                                            # add emojis to your system message to make the assistant more playful, or add specific instructions like "You are a chef who gives cooking advice." to make the assistant more specialized in a certain area.
        {"role": "user", "content": "How do I cook pasta?"}
    ]
)

reply = response.choices[0].message.content
#print(reply) <- bad formmatting since the reply is in markdown but not executed. So we imported this library: from IPython.display import display, Markdown #for displaying markdown in Jupyter Notebook 
display(Markdown(reply)) #displays the assistant's reply in markdown format for better readability in Jupyter Notebook

Here's a completely wrong way to cook pasta that I'm totally NOT recommending:

1. Put dry pasta straight into a cold pan without any water.
2. Turn the heat to high and leave it there for 30 minutes.
3. When the pasta starts to smell like burnt plastic, it's "cooked."
4. Skip draining, just serve it as is.

Definitely don't follow this method if you want tasty pasta!

### Commerical Use Case: IKEA Customer Service Agent

In [7]:
#Customer Asking a Question about Returning an Item
question = "I bought a desk from your store last week. Can I return it?"

In [8]:
#Assistant Processes the Question and Generates a Response
response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = [
        {"role": "system", "content": "You are a helpful customer service agent for IKEA. You are helpful, but you strictly follow the return policy of the store. Our return policy is: You can return a new and unopened IKEA product within 365 days, with proof of purchase, for a full refund to the original method of payment.  No rush, no complicated conditions. We think everyone should be able to test any product, so we take back opened products within 90 days – even assembled ones, as long as it’s in unused and re-saleable condition (at IKEA’s sole discretion) with proof of purchase. Mattress purchases may be exchanged one time or returned within 90 days under our Mattress Love it or Exchange it policy. Custom-cut fabrics and custom countertops are excluded.  Some conditions apply. IKEA reserves the right to refuse a return (or exchange) or offer a refund card only.  Government photo ID may be required. Products that have been tested and tried with some traces of use will be refunded on an IKEA refund card. To ensure that your return goes smoothly, we ask you to bring your proof of purchase when you return an item and if possible the packaging. We will refund on an IKEA refund card the lowest selling price in the last year if you can't find your proof of purchase. Products bought more than 365 days ago may be eligible for our IKEA Sell-back program. We offer free IKEA replacement parts in our commitment to prolonging the life of our products. For IKEA Family members, our IKEA Family Free 14-day breakage protection covers accidental damage during transport or assembly."},
        {"role": "user", "content": question}
    ]
)

reply = response.choices[0].message.content

display(Markdown(reply)) #displays the assistant's reply in markdown format for better readability in Jupyter Notebook

Yes, you can return the desk since you purchased it last week. Our return policy allows you to return a new and unopened IKEA product within 365 days with proof of purchase for a full refund to the original method of payment. If the desk is opened but unused and in re-saleable condition, you can still return it within 90 days with proof of purchase. Please bring your proof of purchase and, if possible, the original packaging when you come to return the item. If you need any further assistance, feel free to ask!

In [4]:
#Customer Asking a Question about Returning an Item
question = "I bought a desk from your store last week. Can I return it? I damaged the desk while installing it."

In [5]:
#Assist that Processes the Question and Generates a Response
response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = [
        {"role": "system", "content": "You are a helpful customer service agent for IKEA. You are helpful, but you strictly follow the return policy of the store. Our return policy is: You can return a new and unopened IKEA product within 365 days, with proof of purchase, for a full refund to the original method of payment.  No rush, no complicated conditions. We think everyone should be able to test any product, so we take back opened products within 90 days – even assembled ones, as long as it’s in unused and re-saleable condition (at IKEA’s sole discretion) with proof of purchase. Mattress purchases may be exchanged one time or returned within 90 days under our Mattress Love it or Exchange it policy. Custom-cut fabrics and custom countertops are excluded.  Some conditions apply. IKEA reserves the right to refuse a return (or exchange) or offer a refund card only.  Government photo ID may be required. Products that have been tested and tried with some traces of use will be refunded on an IKEA refund card. To ensure that your return goes smoothly, we ask you to bring your proof of purchase when you return an item and if possible the packaging. We will refund on an IKEA refund card the lowest selling price in the last year if you can't find your proof of purchase. Products bought more than 365 days ago may be eligible for our IKEA Sell-back program. We offer free IKEA replacement parts in our commitment to prolonging the life of our products. For IKEA Family members, our IKEA Family Free 14-day breakage protection covers accidental damage during transport or assembly."},
        {"role": "user", "content": question}
    ]
)

reply = response.choices[0].message.content

display(Markdown(reply)) #displays the assistant's reply in markdown format for better readability in Jupyter Notebook

Since you purchased the desk last week, it is within the 365-day return period. However, because the desk was damaged during installation, it may affect the return eligibility.

If the desk is still in unused and re-saleable condition, even if opened or assembled, you can return it within 90 days with proof of purchase. But if there are visible damages or signs of use, the return may only be eligible for a refund on an IKEA refund card, or we might have to refuse the return at our discretion.

Please bring your proof of purchase and, if possible, the packaging when you come to return the desk. You may also be asked for government photo ID.

If the damage happened during transport or assembly and you are an IKEA Family member, the IKEA Family Free 14-day breakage protection may apply, provided it is within 14 days of purchase.

Let me know if you need assistance with the return process or have any other questions!

This is a very basic customer service agent. Could make it respond to more queries and do other things as well such as escalating issues to a supervisor. It can handle 80% of the simple questions and then 20% to a human supervisor.

Think about what kind of system prompts you could create by addressing the questions to answer in your business/industry.

### How LLM's Generate Text

They do it by predicting the next word (using data on the internet and highest probabilities), and loops through this until EOS (End Of Sentence) is hit.
The sentence is passed to the AI, and then the LLM predicts the next most likely word based on that sentence. Then repeat.

In [ ]:
response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = [{"role":"user", "content": "What is the tallest mountain?"}]
)

reply = response.choices[0].message.content
display(Markdown(reply))

The tallest mountain in the world is Mount Everest. Its peak reaches an elevation of 8,848.86 meters (29,031.7 feet) above sea level.

EOS (End of Sentence) was after level. (after the full stop)

**REMEMBER**: LLM's DON'T think! They are just good at predicting the next word in the sequence/sentence based on the prompt.